### Core Clustering Logic

In [ ]:
# KMeans
import pandas as pd
import mysql.connector
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='Yoimiya3000!',
    database='student_db'
)

query = """
SELECT s.student_id, s.name,
       c.city, c.state, c.fee,
       d.department_name, r.subject_name, s.skill_name
FROM students e
LEFT JOIN contacts c ON s.student_id = c.student_id
LEFT JOIN compensation comp ON s.student_id = comp.student_id
LEFT JOIN departments d ON s.department_id = d.department_id
LEFT JOIN users u ON s.student_id = u.student_id
LEFT JOIN subjects r ON u.subject_id = r.subject_id
LEFT JOIN student_course es ON s.student_id = es.student_id
LEFT JOIN course s ON es.skill_id = s.skill_id
"""
df = pd.read_sql(query, conn)
conn.close()

# skill_name -> one-hot encoding
skill_dummies = df.pivot_table(index='student_id', 
                               columns='skill_name', 
                               aggfunc='size', 
                               fill_value=0)
df = df.drop('skill_name', axis=1).drop_duplicates(subset='student_id')
df = df.merge(skill_dummies, left_on='student_id', right_index=True, how='left')
df.fillna(0, inplace=True)

# Feature sets for clustering
feature_sets = {
    "fee_only": ['fee'],
    "location_only": ['city', 'state'],
    "dept_only": ['department_name', 'subject_name'],
    "subjects_only": list(skill_dummies.columns),
    "subjects_and_dept": ['fee', 'department_name', 'subject_name'],
    "dept_and_fee": ['fee', 'dept'],
    "all_features": ['fee', 'department_name', 'city', 'state', 'subject_name'] + list(skill_dummies.columns)
}

# Looping and Clustering 
for name, feature_cols in feature_sets.items():
    numeric_feats = [f for f in feature_cols if f == 'fee'] # extracts numeric features
    categorical_feats = [f for f in feature_cols if f in ['department_name', 'city', 'state', 'subject_name']] # extracts categorical features
    
    # Preprocessing 
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_feats),
            ('cat', OneHotEncoder(), categorical_feats)
        ],
        remainder='passthrough'  # any other columns kept as it is
    )
    
    kmeans = KMeans(n_clusters=7, random_state=42)  
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('clusterer', kmeans)
    ])
    
    df[f'cluster_{name}'] = pipelins.fit_predict(df[feature_cols]) + 1 # starting from 1
print(df[['student_id'] + [f'cluster_{name}' for name in feature_sets.keys()]])


In [ ]:
# AGNES 
from sklearn.cluster import AgglomerativeClustering

for name, feature_cols in feature_sets.items():
    numeric_feats = [f for f in feature_cols if f == 'fee']
    categorical_feats = [f for f in feature_cols if f in ['department_name', 'city', 'state', 'subject_name']]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_feats),
            ('cat', OneHotEncoder(sparse_output=False), categorical_feats)
        ],
        remainder='passthrough'
    )

    agg = AgglomerativeClustering(n_clusters=7)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('clusterer', agg)
    ])

    df[f'cluster_agg_{name}'] = pipelins.fit_predict(df[feature_cols]) + 1

print(df[['student_id'] + [f'cluster_agg_{name}' for name in feature_sets.keys()]])


In [ ]:
# Gaussian
from sklearn.mixture import GaussianMixture

for name, feature_cols in feature_sets.items():
    numeric_feats = [f for f in feature_cols if f == 'fee']
    categorical_feats = [f for f in feature_cols if f in ['department_name', 'city', 'state', 'subject_name']]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_feats),
            ('cat', OneHotEncoder(sparse_output=False), categorical_feats)
        ],
        remainder='passthrough'
    )

    gmm = GaussianMixture(n_components=7, random_state=42)

    X_transformed = preprocessor.fit_transform(df[feature_cols])
    df[f'cluster_gmm_{name}'] = gmm.fit_predict(X_transformed) + 1

print(df[['student_id'] + [f'cluster_gmm_{name}' for name in feature_sets.keys()]])
